# Setup

In [ ]:
%pip install colab-xterm
%load_ext colabxterm

In [ ]:
# Run in terminal
# curl -sSL https://ollama.ai/install.sh | sh && ollama serve

In [ ]:
%xterm

In [ ]:
! curl http://localhost:11434/api/pull -d '{  "model": "gemma3:4b" }'

In [ ]:
%pip install -q llama-index-llms-ollama llama-index

Kod instaluje dwie biblioteki Pythona przy użyciu menedżera pakietów pip:

1. llama-index-llms-openai - zawiera interfejsy do komunikacji z modelami językowymi OpenAI
2. llama-index - framework do budowania aplikacji wykorzystujących duże modele językowe

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.llms.ollama import Ollama
from llama_index.core.tools import FunctionTool

Ten kod tworzy agenta AI wykorzystującego technikę `ReAct` (Reasoning and Acting) z biblioteką llama_index. Przeanalizujmy go linia po linii:

Pierwsze cztery linijki importują niezbędne komponenty:
```python
from llama_index.core.agent import ReActAgent
from llama_index.llms.ollama import Ollama
from llama_index.core.tools import FunctionTool
```

Te importy dotyczą:
- `ReActAgent` - klasa implementująca agenta, który łączy rozumowanie z działaniem
- `Ollama` - interfejs do lokalnych modeli językowych uruchamianych przez narzędzie Ollama
- `FunctionTool` - klasa do definiowania narzędzi, które agent może wykorzystywać

In [ ]:
class CFG:
    model = "gemma3:4b"

# Funkcje


In [4]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers and returns the result integer"""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)

In [ ]:
def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


add_tool = FunctionTool.from_defaults(fn=add)

# Query


In [ ]:
llm = Ollama(model=CFG.model, request_timeout=60.0)

In [ ]:
agent = ReActAgent.from_tools(
    [multiply_tool, add_tool], llm=llm, verbose=True, max_iterations=30
)

Ta linia kodu tworzy agenta ReAct, który będzie wykorzystywał modele językowe do rozumowania i podejmowania działań. Analizując szczegółowo:

```python
agent = ReActAgent.from_tools([multiply_tool, add_tool], llm=llm, verbose=True, max_iterations=30)
```

Jest to moment, gdy wszystkie wcześniej zdefiniowane elementy łączą się w funkcjonalnego agenta AI. Rozbijmy tę linię na poszczególne komponenty:

1. `agent` - zmienna, która będzie przechowywać stworzonego agenta. Ten obiekt będzie głównym punktem interakcji z systemem.

2. `ReActAgent.from_tools(...)` - statyczna metoda fabryczna klasy `ReActAgent`, która tworzy nową instancję agenta. Nazwa "ReAct" pochodzi od połączenia słów "Reasoning" (rozumowanie) i "Acting" (działanie), co odzwierciedla dwuetapowy proces: agent najpierw rozumuje o problemie, a następnie podejmuje konkretne działania.

3. `[multiply_tool, add_tool]` - lista narzędzi, które zostają przekazane agentowi. Są to dokładnie te narzędzia, które zdefiniowaliśmy wcześniej - `multiply_tool` do mnożenia liczb całkowitych i `add_tool` do ich dodawania. Agent będzie mógł wybierać pomiędzy tymi narzędziami podczas rozwiązywania problemów.

4. `llm=llm` - przekazuje instancję modelu językowego Ollama, którą utworzyliśmy w poprzednim kroku. Ten model będzie "mózgiem" agenta, odpowiedzialnym za rozumowanie, planowanie i decyzje.

5. `verbose=True` - włącza tryb szczegółowego wypisywania informacji. Gdy parametr jest ustawiony na `True`, agent będzie pokazywał każdy krok swojego procesu rozumowania i działania, co jest bardzo przydatne podczas debugowania i zrozumienia, jak agent dochodzi do swoich wniosków.

6. `max_iterations=30` - ustala maksymalną liczbę kroków, które agent może wykonać przy rozwiązywaniu pojedynczego problemu. Jest to mechanizm zabezpieczający, który zapobiega nieskończonym pętlom - jeśli agent nie znajdzie rozwiązania w 30 krokach, proces zostanie przerwany. Liczba 30 sugeruje, że autor kodu spodziewa się, że problemy, które będzie rozwiązywał agent, nie są skrajnie złożone, ale mogą wymagać kilku lub kilkunastu kroków rozumowania.

W praktyce, po utworzeniu takiego agenta, możliwe będzie zadawanie mu pytań lub przedstawianie problemów, a on będzie:
1. Analizował problem przy użyciu modelu językowego
2. Decydował, które narzędzie (dodawanie czy mnożenie) jest najbardziej odpowiednie do konkretnego kroku
3. Wywoływał odpowiednie narzędzie z właściwymi parametrami
4. Interpretował wyniki i albo udzielał odpowiedzi, albo kontynuował rozumowanie z uwzględnieniem uzyskanych wyników

Parametr `verbose=True` oznacza, że będziemy mogli obserwować cały ten proces "myślenia" agenta, co jest niezwykle wartościowe zarówno do celów edukacyjnych, jak i dla deweloperów pracujących nad udoskonaleniem systemu.

In [8]:
prompt_dict = agent.get_prompts()
for k, v in prompt_dict.items():
    print(f"Prompt: {k}\n\nValue: {v.template}")

Prompt: agent_worker:system_prompt

Value: You are designed to help with a variety of tasks, from answering questions to providing summaries to other types of analyses.

## Tools

You have access to a wide variety of tools. You are responsible for using the tools in any sequence you deem appropriate to complete the task at hand.
This may require breaking the task into subtasks and using different tools to complete each subtask.

You have access to the following tools:
{tool_desc}


## Output Format

Please answer in the same language as the question and use the following format:

```
Thought: The current language of the user is: (user's language). I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names}) if using a tool.
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

Please ALWAYS start with a Thought.

NEVER surround your response with markdown code markers. You may

In [9]:
response = agent.chat("Jaki jest wynik 20+(2*4)? Przelicz krok po kroku ")

> Running step 4b2b3b3a-3f3c-413a-bf67-cd89f6bd4ae3. Step input: Jaki jest wynik 20+(2*4)? Przelicz krok po kroku 
Thought: I need to evaluate the expression 20 + (2 * 4) step by step. I will use the multiply tool to perform the multiplication and then the add tool to perform the addition.
Action: multiply
Action Input: {'a': 2, 'b': 4}
Observation: 8
> Running step 6d67723a-7624-47fc-9327-abc0a9dd4962. Step input: None
Thought: Now I have the result of the multiplication, which is 8. I need to add 20 to this result. I will use the add tool for this.
Action: add
Action Input: {'a': 20, 'b': 8}
Observation: 28
> Running step 995323d3-2bbd-4fd1-a3f2-88e523c5e507. Step input: None
Thought: I have now performed the addition and the result is 28. I can answer without using any more tools.
Answer: Wynik wyrażenia 20 + (2 * 4) to 28.
